# The Three Tribes of the Palmer Archipelago
### An Interactive Data Exploration Dashboard

*Set in the rugged Palmer Archipelago of Antarctica, this dashboard invites you to investigate how three distinct penguin "tribes" — the **Adélie**, **Chinstrap**, and **Gentoo** — navigate territory, physical evolution, and biological competition.*

> **How to use:** Run each cell in order (Shift+Enter). Interactive widgets will appear below code cells — use them to explore the data dynamically.

---

## Setup & Configuration

In [1]:
# ── Imports ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output
import warnings
warnings.filterwarnings("ignore")

# ── Color Palettes (OHA-compliant) ─────────────────────────
SPECIES_COLORS = {
    'Adelie': '#15478A',     # Midnight Blue
    'Chinstrap': '#5BB5D5',  # Viking
    'Gentoo': '#3B5BBE'      # Electric Indigo
}
SPECIES_ORDER = ['Adelie', 'Chinstrap', 'Gentoo']

# OHA Rule §5: Sex encoding
SEX_COLORS = {'Male': '#419164', 'Female': '#876EC4'}
TEMPLATE = 'plotly_white'
GRIDCOLOR = '#D3D3D3'
SOURCE_TEXT = 'Source: Palmer Penguins Dataset (Horst, Hill & Gorman, 2020)'

# ── Global Layout Defaults (OHA Rule §3) ───────────────────
LAYOUT_DEFAULTS = dict(
    font_family='Arial',
    title_font_size=14,
    title_font_color='#202020',
    plot_bgcolor='rgba(0,0,0,0)',
)

def source_annotation():
    return dict(x=1, y=-0.18, xref='paper', yref='paper', showarrow=False,
                text=SOURCE_TEXT, font=dict(size=9, color='#909090'), xanchor='right')

# ── Narrative Styling Helpers ──────────────────────────────
def narrative_box(text):
    display(HTML(
        '<div style="background: linear-gradient(135deg, #fdfcfb 0%, #e2d1c3 100%);'
        'border-left: 5px solid #15478A; padding: 16px 22px; margin: 14px 0;'
        'border-radius: 0 10px 10px 0; font-style: italic; color: #333;'
        'font-size: 14px; line-height: 1.7; font-family: Arial, sans-serif;'
        'box-shadow: 0 2px 8px rgba(0,0,0,0.06);">'
        + text + '</div>'
    ))

def insight_box(text):
    display(HTML(
        '<div style="background: #F9F9F9; border-left: 5px solid #15478A; '
        'padding: 16px 22px; margin: 14px 0; border-radius: 0 4px 4px 0; '
        'color: #333; font-size: 14px; line-height: 1.6; font-family: Arial;">'
        + '<strong>Key Insight:</strong> ' + text + '</div>'
    ))

print("Setup complete. All libraries loaded and OHA styling configured.")

Setup complete. All libraries loaded and OHA styling configured.


## Data Loading & Preprocessing

In [2]:
# ── Load and Clean the Palmer Penguins dataset ─────────────
df_raw = sns.load_dataset('penguins')
df = df_raw.dropna().reset_index(drop=True)
df['sex'] = df['sex'].str.capitalize()

print(f"Raw dataset -> Cleaned dataset: {len(df)} complete observations retained.")

Raw dataset -> Cleaned dataset: 333 complete observations retained.


In [3]:
# ── Sample Data View ───────────────────────────────────────
display(Markdown("### Sample Data"))
display(
    df.head(6).style
    .format({'bill_length_mm': '{:.1f}', 'bill_depth_mm': '{:.1f}', 
             'flipper_length_mm': '{:.0f}', 'body_mass_g': '{:,.0f}'})
    .set_properties(**{'font-family': 'Arial'})
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white')]}])
)

### Sample Data

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181,"3,750",Male
1,Adelie,Torgersen,39.5,17.4,186,"3,800",Female
2,Adelie,Torgersen,40.3,18.0,195,"3,250",Female
3,Adelie,Torgersen,36.7,19.3,193,"3,450",Female
4,Adelie,Torgersen,39.3,20.6,190,"3,650",Male
5,Adelie,Torgersen,38.9,17.8,181,"3,625",Female


In [4]:
# ── Summary Statistics by Species ──────────────────────────
display(Markdown("### Tribe Profiles"))
summary = df.groupby('species').agg(
    Count=('species', 'count'),
    Avg_Bill_Length=('bill_length_mm', 'mean'),
    Avg_Bill_Depth=('bill_depth_mm', 'mean'),
    Avg_Flipper_Length=('flipper_length_mm', 'mean'),
    Avg_Body_Mass=('body_mass_g', 'mean')
).round(1)

display(
    summary.style
    #.background_gradient(cmap='YlGnBu', axis=0)
    #.set_table_styles([{'selector': 'th', 'props': [('background-color', '#15478A'), ('color', 'white')]}])
    .format({'Avg_Body_Mass': '{:,.0f}', 'Avg_Bill_Length': '{:.1f}',
             'Avg_Bill_Depth': '{:.1f}', 'Avg_Flipper_Length': '{:.0f}'})
)

### Tribe Profiles

,Count,Avg_Bill_Length,Avg_Bill_Depth,Avg_Flipper_Length,Avg_Body_Mass
species,,,,,
Adelie,146,38.8,18.3,190,"3,706"
Chinstrap,68,48.8,18.4,196,"3,733"
Gentoo,119,47.6,15.0,217,"5,092"


In [5]:
# ── OVERALL COMPOSITIONAL OVERVIEW ─────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Retrieve counts; filtering out N/A genders to match your clean image percentages
sp_counts = df['species'].value_counts()
sex_counts = df['sex'].dropna().value_counts()

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]], 
                    subplot_titles=['Species Distribution', 'Gender Distribution'])

# 1. Left Pie: Species
sp_colors = [SPECIES_COLORS.get(sp, '#000') for sp in sp_counts.index]
fig.add_trace(go.Pie(
    labels=sp_counts.index, 
    values=sp_counts.values,
    name="Species",
    marker=dict(colors=sp_colors),
    textinfo='percent+value', 
    texttemplate="%{percent:.1%} - %{value}",
    hoverinfo='label+percent',
    textposition='inside',
    showlegend=False # Disable default merged legend
), 1, 1)

# 2. Right Pie: Gender
gender_colors = {'Female': '#FF4488', 'Male': '#3377FF'}
g_colors = [gender_colors.get(g, '#cccccc') for g in sex_counts.index]
fig.add_trace(go.Pie(
    labels=sex_counts.index, 
    values=sex_counts.values,
    name="Gender",
    marker=dict(colors=g_colors),
    textinfo='percent+value',
    texttemplate="%{percent:.1%} - %{value}",
    hoverinfo='label+percent',
    textposition='inside',
    showlegend=False # Disable default merged legend
), 1, 2)


# --- Build Independent Custom Legends ---

# Custom Legend 1 (Species) mapped to layout 'legend'
for sp, color in zip(sp_counts.index, sp_colors):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(color=color, size=15, symbol='square'),
        name=sp, legend='legend'
    ))

# Custom Legend 2 (Gender) mapped to layout 'legend2'
for g, color in zip(sex_counts.index, g_colors):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(color=color, size=15, symbol='square'),
        name=g, legend='legend2'
    ))


# --- Place the Split Legends Independently ---
fig.update_layout(
    **LAYOUT_DEFAULTS,
    title=dict(
        text='<span style="font-size:24px; text-decoration: underline;"><b>Overall Distribution and Island Filtering</b></span><br><br><span style="font-size:28px;"><b>Archipelago Overview</b></span>',
        x=0.5, xanchor='center'
    ),
    height=500,
    margin=dict(t=120, b=20, l=20, r=20),
    
    # Position Species legend anchored to the bottom right of the LEFT quadrant
    legend=dict(
        title="Species",
        orientation="v", 
        yanchor="bottom", y=0.05, 
        xanchor="right", x=0.48,
        bgcolor='rgba(255,255,255,0.7)'
    ),
    
    # Position Gender legend anchored to the bottom right of the RIGHT quadrant
    legend2=dict(
        title="Gender",
        orientation="v", 
        yanchor="bottom", y=0.05, 
        xanchor="right", x=1.0,
        bgcolor='rgba(255,255,255,0.7)'
    )
)

fig.show()


---
# Act 1: The Territory
## Geographic Distribution of the Three Tribes

> *"Set the stage by exploring where these tribes live. While the Adélies are hardy 'pioneers' found on every island, the Gentoos and Chinstraps are more exclusive to their specific territories."*

In [6]:
narrative_box(
    "<strong>The Territory:</strong> Before we understand these penguin tribes, "
    "we must first understand their land. The Palmer Archipelago consists of three islands — "
    "<strong>Biscoe</strong>, <strong>Dream</strong>, and <strong>Torgersen</strong> — "
    "each with its own ecological character. Where a species chooses to live tells us much "
    "about its survival strategy."
)

In [7]:
# ── Archipelago Overview: Stacked by Sex ───────────────────

def plot_territory(island_filter='All Islands'):
    #clear_output(wait=True)
    if island_filter == 'All Islands':
        island_stats = df.groupby(['island', 'species']).size().reset_index(name='count')
        islands_list = sorted(df['island'].unique())
        
        x_isl = []
        x_sp = []
        y_count = []
        colors = []
        
        for isl in islands_list:
            for sp in SPECIES_ORDER:
                val = island_stats[(island_stats['island'] == isl) & (island_stats['species'] == sp)]['count'].values
                count = val[0] if len(val) > 0 else 0
                x_isl.append(isl.upper())
                x_sp.append(sp)
                y_count.append(count)
                colors.append(SPECIES_COLORS[sp])
                
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=[x_isl, x_sp],
            y=y_count,
            marker_color=colors,
            text=[c if c > 0 else '' for c in y_count],
            textposition='outside',
            textfont=dict(color='#505050', size=11),
            showlegend=False
        ))
        
        for sp in SPECIES_ORDER:
            fig.add_trace(go.Bar(
                x=[[None], [None]], y=[None],
                name=sp, marker_color=SPECIES_COLORS[sp],
                showlegend=True
            ))

        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title=dict(text='ADELIES ARE THE ONLY REGIONAL PIONEERS<br>'
                            '<sup style="color:#505050;">Populations grouped by Island, segmented by Species</sup>'),
            height=450,
            margin=dict(t=100, b=60),
            legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02, borderwidth=0),
            annotations=[source_annotation()]
        )
        fig.update_xaxes(showgrid=False, linecolor='black')
        fig.update_yaxes(title='Population', showgrid=True, gridcolor=GRIDCOLOR)
        fig.show()

    else:
        island_stats = df[df['island'] == island_filter].groupby(['species', 'sex']).size().reset_index(name='count')
        
        x_sp = []
        y_male = []
        y_female = []
        y_total = []
        
        for sp in SPECIES_ORDER:
            m_val = island_stats[(island_stats['species'] == sp) & (island_stats['sex'] == 'Male')]['count'].values
            f_val = island_stats[(island_stats['species'] == sp) & (island_stats['sex'] == 'Female')]['count'].values
            
            m_count = m_val[0] if len(m_val) > 0 else 0
            f_count = f_val[0] if len(f_val) > 0 else 0
            
            x_sp.append(sp)
            y_male.append(m_count)
            y_female.append(f_count)
            y_total.append(m_count + f_count)
            
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            x=x_sp, y=y_male, name='Male',
            marker_color=SEX_COLORS['Male'],
            text=[m if m > 0 else '' for m in y_male],
            textposition='inside', textfont=dict(color='white')
        ))
        fig.add_trace(go.Bar(
            x=x_sp, y=y_female, name='Female',
            marker_color=SEX_COLORS['Female'],
            text=[f if f > 0 else '' for f in y_female],
            textposition='inside', textfont=dict(color='white')
        ))
        
        fig.add_trace(go.Scatter(
            x=x_sp, y=y_total, mode='text',
            text=[t if t > 0 else '' for t in y_total],
            textposition='top center', textfont=dict(color='#505050', size=11),
            showlegend=False, hoverinfo='skip'
        ))
        
        fig.update_layout(
            **LAYOUT_DEFAULTS,
            title=dict(text=f'{island_filter.upper()} ISLAND POPULATION DYNAMICS<br>'
                            f'<sup style="color:#505050;">Species segmented and stacked vertically by Sex</sup>'),
            height=450, barmode='stack',
            margin=dict(t=100, b=60),
            legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02, borderwidth=0),
            annotations=[source_annotation()]
        )
        fig.update_xaxes(showgrid=False, linecolor='black')
        fig.update_yaxes(title='Population', showgrid=True, gridcolor=GRIDCOLOR)
        fig.show()

island_dropdown = widgets.Dropdown(
    options=['All Islands'] + sorted(df['island'].unique().tolist()),
    value='All Islands',
    description='Island:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
display(Markdown("### Filter by island to explore territorial shifts:"))
out = widgets.interactive_output(plot_territory, {'island_filter': island_dropdown})
display(island_dropdown, out)

insight_box("Adelie is the only species situated across all three islands — true ecological pioneers. "
            "Gentoo strictly resides on Biscoe, while Chinstraps are exclusive to Dream. "
            "Additionally, the stacks perfectly demonstrate the even 50/50 gender survival ratio per colony.")

### Filter by island to explore territorial shifts:

Dropdown(description='Island:', layout=Layout(width='280px'), options=('All Islands', 'Biscoe', 'Dream', 'Torg…

Output()

---
# Act 2: The Physical Paradox
## Simpson's Paradox in Morphology

> *"Appearances can be deceiving. This act reveals a 'hidden truth': while the overall data might suggest one trend, looking at individual tribes reveals the true biological relationships."*

In [8]:
narrative_box(
    "<strong>The Paradox:</strong> Simpson's Paradox is one of the most fascinating "
    "phenomena in statistics — a trend that appears in several groups of data "
    "<em>reverses</em> when the groups are combined. The penguin bill data contains "
    "a textbook-perfect example. Watch what happens when we reveal the tribes..."
)

In [9]:
# ── Simpson's Paradox: Interactive Feature Explorer ──────────

def plot_simpsons_paradox(show_species=False, compare_feature='Bill Length'):
    clear_output(wait=True)
    col_map = {
        'Bill Length': 'bill_length_mm',
        'Flipper Length': 'flipper_length_mm',
        'Body Mass': 'body_mass_g'
    }
    unit_map = {
        'Bill Length': '(mm)',
        'Flipper Length': '(mm)',
        'Body Mass': '(g)'
    }
    
    x_col = col_map[compare_feature]
    x_unit = unit_map[compare_feature]
    y_col = 'bill_depth_mm'
    
    fig = go.Figure()

    if not show_species:
        # ── MISLEADING: All data as one group ─
        clean_df = df.dropna(subset=[x_col, y_col])
        fig.add_trace(go.Scatter(
            x=clean_df[x_col], y=clean_df[y_col],
            mode='markers',
            marker=dict(color='#8C8C91', size=8, opacity=0.5),
            name='All Penguins',
            hovertemplate=f'{compare_feature}: %{{x:.1f}}{x_unit}<br>Bill Depth: %{{y:.1f}}mm<extra></extra>'
        ))
        
        if len(clean_df) > 1:
            z = np.polyfit(clean_df[x_col], clean_df[y_col], 1)
            x_line = np.linspace(clean_df[x_col].min(), clean_df[x_col].max(), 100)
            fig.add_trace(go.Scatter(
                x=x_line, y=np.polyval(z, x_line), mode='lines',
                line=dict(color='#dc3545', width=2.5, dash='dash'),
                name=f'Overall Trend (slope = {z[0]:.3f})',
            ))
            
        title = f'OVERALL DATA SUGGESTS A NEGATIVE {compare_feature.upper()} CORRELATION'
        subtitle = f'With no groups, greater {compare_feature.lower()} appears to have shallower depth — but is this real?'

    else:
        # ── TRUTH: Colored by species ──────────────────────
        for species in SPECIES_ORDER:
            sp = df[df['species'] == species].dropna(subset=[x_col, y_col])
            fig.add_trace(go.Scatter(
                x=sp[x_col], y=sp[y_col],
                mode='markers',
                marker=dict(color=SPECIES_COLORS[species], size=8, opacity=0.7),
                name=species,
                hovertemplate=(f'<b>{species}</b><br>'
                               f'{compare_feature}: %{{x:.1f}}{x_unit}<br>'
                               f'Bill Depth: %{{y:.1f}}mm<extra></extra>')
            ))
            if len(sp) > 1:
                z = np.polyfit(sp[x_col], sp[y_col], 1)
                x_line = np.linspace(sp[x_col].min(), sp[x_col].max(), 100)
                fig.add_trace(go.Scatter(
                    x=x_line, y=np.polyval(z, x_line), mode='lines',
                    line=dict(color=SPECIES_COLORS[species], width=2),
                    name=f'{species} (slope = {z[0]:.3f})',
                    showlegend=False,
                ))
        title = 'EACH SPECIES ACTUALLY SHOWS A POSITIVE CORRELATION'
        subtitle = f"Within every tribe, greater {compare_feature.lower()} means deeper bill depth — Simpson's Paradox revealed."

    fig.update_layout(
        **LAYOUT_DEFAULTS,
        title=dict(text=f'{title}<br><sup style="color:#505050;font-weight:normal">{subtitle}</sup>'),
        xaxis_title=f'{compare_feature} {x_unit}', yaxis_title='Bill Depth (mm)',
        height=560,
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)', borderwidth=0),
        margin=dict(b=80),
        annotations=[source_annotation()],
    )
    fig.update_xaxes(showgrid=True, gridcolor=GRIDCOLOR)
    fig.update_yaxes(showgrid=True, gridcolor=GRIDCOLOR)
    fig.show()

# Interactive Widgets
feature_dropdown = widgets.Dropdown(
    options=['Bill Length', 'Flipper Length', 'Body Mass'],
    value='Bill Length',
    description='Compare against:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

toggle = widgets.ToggleButton(
    value=False,
    description='Reveal the Tribes!',
    button_style='warning',
    tooltip="Toggle species coloring to see the paradox",
    icon='eye',
    layout=widgets.Layout(width='260px', height='42px'),
)

# Use an HBox to put the dropdown and toggle side-by-side
controls = widgets.HBox([feature_dropdown, toggle])

display(Markdown("### Expand the paradox across multiple traits:"))
out = widgets.interactive_output(plot_simpsons_paradox, {'show_species': toggle, 'compare_feature': feature_dropdown})
display(controls, out)

### Expand the paradox across multiple traits:

Output()

In [10]:
insight_box(
    "When viewed as a single population, bill length and bill depth appear "
    "<strong>negatively correlated</strong> (slope ~ -0.065). But when separated by species, "
    "each tribe shows a <strong>positive correlation</strong>! The Gentoos, with their uniquely "
    "long but shallow bills, shift the overall trend downward — a textbook case of "
    "<strong>Simpson's Paradox</strong>."
)

In [11]:
insight_box(
    "When viewed as a single population, bill depth falsely appears to scale negatively with length, flippers, and body mass. " 
    "But isolating by species proves that within each tribe, positive evolutionary morphology is strictly maintained."
)

---
# Act 3: The Engine of the Penguin
## Visualizing Core Scaling and Profiling Athletes

> *"Which physical attribute dictates the overall weight of a penguin? We start purely through visual observation across dimensions, mathematically validate it using heatmaps, and deduce the profiles."*

In [12]:
# ── Step 1: Visual Hypothesis (Scatter Grid & Bar Charts) ──────────

narrative_box("<strong>The Discovery Phase:</strong> We map Body Mass against the three other core bodily mechanics. Which scatter cloud yields the tightest linear fit?")

def plot_visual_hypothesis(metric_pair='Mass vs Flipper Length'):
    clear_output(wait=True)
    if metric_pair == 'Mass vs Flipper Length':
        xcol = 'flipper_length_mm'
        xname = 'Flipper Length'
    elif metric_pair == 'Mass vs Bill Length':
        xcol = 'bill_length_mm'
        xname = 'Bill Length'
    elif metric_pair == 'Mass vs Bill Depth':
        xcol = 'bill_depth_mm'
        xname = 'Bill Depth'
        
    fig = make_subplots(
        rows=2, cols=2,
        column_widths=[0.65, 0.35],
        specs=[[{"rowspan": 2}, {}],
               [None, {}]],
        subplot_titles=[
            f'Body Mass vs {xname}', 
            'Average Body Mass (g)', 
            f'Average {xname} (mm)'
        ],
        horizontal_spacing=0.1,
        vertical_spacing=0.15
    )

    for sp in SPECIES_ORDER:
        d_sp = df[df['species'] == sp]
        fig.add_trace(go.Scatter(
            x=d_sp[xcol], y=d_sp['body_mass_g'], mode='markers',
            marker=dict(color=SPECIES_COLORS[sp], size=6, opacity=0.7),
            name=sp, showlegend=True, legendgroup=sp,
            hovertemplate=f'<b>{sp}</b><br>{xname}: %{{x}}<br>Mass: %{{y}} g<extra></extra>'
        ), row=1, col=1)

    avg_mass = df.groupby('species')['body_mass_g'].mean().reindex(SPECIES_ORDER)
    for sp in SPECIES_ORDER:
        fig.add_trace(go.Bar(
            x=[sp], y=[avg_mass[sp]], 
            marker_color=SPECIES_COLORS[sp],
            text=[f"{avg_mass[sp]:,.0f}"], textposition='auto',
            name=sp, showlegend=False, legendgroup=sp,
            hovertemplate=f'<b>{sp}</b><br>Avg Mass: %{{y}} g<extra></extra>'
        ), row=1, col=2)

    avg_x = df.groupby('species')[xcol].mean().reindex(SPECIES_ORDER)
    for sp in SPECIES_ORDER:
        fig.add_trace(go.Bar(
            x=[sp], y=[avg_x[sp]], 
            marker_color=SPECIES_COLORS[sp],
            text=[f"{avg_x[sp]:.1f}"], textposition='auto',
            name=sp, showlegend=False, legendgroup=sp,
            hovertemplate=f'<b>{sp}</b><br>Avg {xname}: %{{y}}<extra></extra>'
        ), row=2, col=2)

    fig.update_layout(
        **LAYOUT_DEFAULTS,
        title=dict(text=f'TRIBAL DISTRIBUTION: {metric_pair.upper()}<br>'
                        '<sup style="color:#505050;">Scatter overview with targeted metrics breakdown</sup>'),
        height=550,
        margin=dict(b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5, borderwidth=0),
        annotations=[source_annotation()]
    )
    
    fig.update_xaxes(title_text=xname + ' (mm)', row=1, col=1)
    fig.update_yaxes(title_text='Body Mass (g)', row=1, col=1)
    fig.update_yaxes(showgrid=True, gridcolor=GRIDCOLOR, row=1, col=2)
    fig.update_yaxes(showgrid=True, gridcolor=GRIDCOLOR, row=2, col=2)
    fig.show()

metric_dropdown = widgets.Dropdown(
    options=['Mass vs Flipper Length', 'Mass vs Bill Length', 'Mass vs Bill Depth'],
    value='Mass vs Flipper Length',
    description='Metric:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
display(Markdown("### Select a metric to explore scaling laws:"))
widgets.interact(plot_visual_hypothesis, metric_pair=metric_dropdown)

### Select a metric to explore scaling laws:

interactive(children=(Dropdown(description='Metric:', layout=Layout(width='280px'), options=('Mass vs Flipper …

<function __main__.plot_visual_hypothesis(metric_pair='Mass vs Flipper Length')>

In [13]:
# ── Step 2: Validation via Species Heatmaps ────────────────

narrative_box("<strong>The Validation Phase:</strong> We compute strictly mathematically. Does our visual assumption regarding the Flipper Engine hold true unconditionally across all three separate species?")

def plot_heatmap(tribe_filter='All Tribes'):
    clear_output(wait=True)
    numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
    nice_labels = ['Bill L', 'Bill D', 'Flipper L', 'Mass']
    
    if tribe_filter == 'All Tribes':
        corr = df[numeric_cols].corr()
        title_text = 'OVERALL POPULATION CORRELATION HEATMAP'
    else:
        corr = df[df['species'] == tribe_filter][numeric_cols].corr()
        title_text = f'INTRA-SPECIES CORRELATION HEATMAP: {tribe_filter.upper()}'
        
    fig = go.Figure(data=go.Heatmap(
        z=corr.values, x=nice_labels, y=nice_labels,
        colorscale='Blues', zmin=-1, zmax=1,
        text=np.round(corr.values, 2), texttemplate="%{text}",
        colorbar=dict(title="corr")
    ))
    
    fig.update_layout(
        **LAYOUT_DEFAULTS, 
        title=dict(text=title_text + '<br>'
                        '<sup style="color:#505050;">Pearson correlation matrix validating evolutionary morphology</sup>'),
        height=450, width=550, margin=dict(b=60),
        annotations=[source_annotation()]
    )
    fig.show()

heatmap_dropdown = widgets.Dropdown(
    options=['All Tribes', 'Adelie', 'Chinstrap', 'Gentoo'],
    value='All Tribes',
    description='Select Tribe:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
display(Markdown("### Filter heatmaps by individual tribes:"))
widgets.interact(plot_heatmap, tribe_filter=heatmap_dropdown)

### Filter heatmaps by individual tribes:

interactive(children=(Dropdown(description='Select Tribe:', layout=Layout(width='280px'), options=('All Tribes…

<function __main__.plot_heatmap(tribe_filter='All Tribes')>

In [14]:
# ── Step 3: Dynamic Text Visualization: Athlete Profiles ──
display(Markdown("### The Findings: Athlete Profiles based on the underlying 'Engine'"))
profiles = {
    'Adelie': ('The Hardy Pioneer', 'Compact and resilient. Found on every island, the Adelie is the generalist of the archipelago — smaller but widespread.'),
    'Chinstrap': ('The Agile Specialist', 'Lean and quick. Occupies a narrow ecological niche on Dream Island.'),
    'Gentoo': ('The Heavyweight Champion', 'The largest of the three tribes. Gentoos boast the longest flippers and the greatest body mass — true powerhouses.')
}

for species in SPECIES_ORDER:
    sp = df[df['species'] == species]
    title, desc = profiles[species]
    color = SPECIES_COLORS[species]

    html = (
        f'<div style="background: white; border-left: 4px solid {color}; border-radius: 4px;'
        f' padding: 16px 22px; margin: 8px 0; font-family: Arial, sans-serif;">'
        f'<div style="font-size: 14px; font-weight: bold; color: {color}; margin-bottom: 4px;">'
        f'{species} — {title}</div>'
        f'<div style="color: #505050; font-size: 12px; margin-bottom: 8px; font-style: italic;">'
        f'{desc}</div>'
        f'<table style="width: 100%; font-size: 12px; color: #333;"><tr>'
        f'<td><strong>Population:</strong> {len(sp)}</td>'
        f'<td><strong>Avg Mass:</strong> {sp["body_mass_g"].mean():,.0f} g</td>'
        f'<td><strong>Islands:</strong> {", ".join(sorted(sp["island"].unique()))}</td>'
        f'<td><strong>Avg Flipper:</strong> {sp["flipper_length_mm"].mean():.1f} mm</td>'
        f'</tr></table></div>'
    )
    display(HTML(html))

insight_box("Because the Flipper functions as the engine, biological necessity requires gentoos to house heavily scaled body masses purely to operate their massive limbs underneath Antarctic currents.")

### The Findings: Athlete Profiles based on the underlying 'Engine'

Population: 146,"Avg Mass: 3,706 g","Islands: Biscoe, Dream, Torgersen",Avg Flipper: 190.1 mm


Population: 68,"Avg Mass: 3,733 g",Islands: Dream,Avg Flipper: 195.8 mm


Population: 119,"Avg Mass: 5,092 g",Islands: Biscoe,Avg Flipper: 217.2 mm


---
# Act 4: The Internal Divide
## Sexual Dimorphism Across All Tribes

> *"Within every tribe, there is a clear divide. This act explores how male penguins consistently outsize females across every metric, a key factor in their social and survival structures."*

In [15]:
narrative_box(
    "<strong>The Internal Divide:</strong> Nature's division runs deep. Within each species, "
    "males consistently outsize females — a phenomenon called <em>sexual dimorphism</em>. "
    "But does this gap remain constant across all physical traits, or do some features "
    "show a bigger divide than others? Use the dropdown below to investigate."
)

In [16]:
# ── Dumbbell Plot for Sexual Dimorphism ────────────────────
def dumbbell_plot(metric='body_mass_g', metric_name='Body Mass (g)'):
    clear_output(wait=True)
    fig = go.Figure()
    stats = df.groupby(['species', 'sex'])[metric].mean().unstack()
    
    for idx, sp in enumerate(SPECIES_ORDER):
        m_val, f_val = stats.loc[sp, 'Male'], stats.loc[sp, 'Female']
        
        # The Handle
        fig.add_trace(go.Scatter(
            x=[f_val, m_val], y=[sp, sp], mode='lines', 
            line=dict(color='#D3D3D3', width=4), showlegend=False
        ))
        # Female Point
        fig.add_trace(go.Scatter(
            x=[f_val], y=[sp], mode='markers', 
            marker=dict(color=SEX_COLORS['Female'], size=14),
            name='Female' if idx == 0 else '', showlegend=(idx==0)
        ))
        # Male Point
        fig.add_trace(go.Scatter(
            x=[m_val], y=[sp], mode='markers', 
            marker=dict(color=SEX_COLORS['Male'], size=14),
            name='Male' if idx == 0 else '', showlegend=(idx==0)
        ))
        
    fig.update_layout(
        **LAYOUT_DEFAULTS,
        title=dict(text=f'MALES OUTSIZE FEMALES CONSISTENTLY ACROSS ALL SPECIES<br>'
                        f'<sup style="color:#505050;">Gender gap (dumbbell plot) for {metric_name}</sup>'),
        height=320, margin=dict(b=80, l=100),
        xaxis_title=metric_name,
        yaxis=dict(categoryorder='array', categoryarray=SPECIES_ORDER[::-1]),
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5, borderwidth=0),
        annotations=[source_annotation()]
    )
    fig.update_xaxes(showgrid=True, gridcolor=GRIDCOLOR)
    fig.update_yaxes(showgrid=False)
    fig.show()
    
    # ── Textual visualization component requirement ────────
    print(f"\nGender Gap Analysis Prints — Metric: {metric_name}")
    for sp in SPECIES_ORDER:
        m = stats.loc[sp, 'Male']
        f_ = stats.loc[sp, 'Female']
        gap = (m - f_) / f_ * 100
        bar_vis = '|' * int(abs(gap))
        print(f"  {sp:>10}: Male {m:>8.1f}  |  Female {f_:>8.1f}  ->  {gap:>+5.1f}%  {bar_vis}")

metrics = {'Body Mass (g)': 'body_mass_g', 'Flipper Length (mm)': 'flipper_length_mm', 
           'Bill Depth (mm)': 'bill_depth_mm', 'Bill Length (mm)': 'bill_length_mm'}
dropdown = widgets.Dropdown(options=metrics.keys(), value='Body Mass (g)', description='Metric:')
display(Markdown("### Select a physical metric to examine the gender gap:"))
widgets.interact(lambda selected: dumbbell_plot(metrics[selected], selected), selected=dropdown)

### Select a physical metric to examine the gender gap:

interactive(children=(Dropdown(description='Metric:', options=('Body Mass (g)', 'Flipper Length (mm)', 'Bill D…

<function __main__.<lambda>(selected)>

---
# Act 5: The Machine Learning Insight
## Clustering & Dimensionality Reduction

> *"Can a computer 'find' the tribes without being told their names? The finale uses unsupervised learning to see if the physical data alone is enough to recreate the species groupings we see in nature."*

In [17]:
narrative_box(
    "<strong>The Machine's Eye:</strong> We utilize purely unsupervised machine learning algorithms to deduce species mathematically. "
    "We specifically execute PCA against four isolated features: <em>Bill Length, Bill Depth, Flipper Length, and Body Mass</em>. "
    "By dropping categorical data (Islands, Sex), we mandate the model to find underlying structural biological patterns "
    "rather than relying on biased geographical coordinates."
)

In [18]:
# ── PCA Dimensional Reduction (2D and 3D) ─────────────────
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X_scaled = StandardScaler().fit_transform(df[features])
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

df_pca = df.copy()
df_pca['PC1'], df_pca['PC2'], df_pca['PC3'] = X_pca[:, 0], X_pca[:, 1], X_pca[:, 2]
var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
var3 = pca.explained_variance_ratio_[2] * 100

fig_2d = px.scatter(
    df_pca, x='PC1', y='PC2', color='species', color_discrete_map=SPECIES_COLORS,
    category_orders={'species': SPECIES_ORDER}, template=TEMPLATE,
)
fig_2d.update_traces(marker=dict(size=8, opacity=0.8), marker_line=dict(width=0.5, color='white'))
fig_2d.update_layout(
    **LAYOUT_DEFAULTS,
    title=dict(text='PCA EXPOSES THREE DISTINCT CLUSTERS IN RAW MEASUREMENTS<br>'
                    f'<sup style="color:#505050;">2D projection of the 4 continuous variables ({var1+var2:.0f}% variance explained)</sup>'),
    height=500, margin=dict(b=80),
    legend=dict(title='', orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5, borderwidth=0),
)
fig_2d.update_xaxes(showline=True, linewidth=1, linecolor='black', gridcolor=GRIDCOLOR, zeroline=False)
fig_2d.update_yaxes(showline=True, linewidth=1, linecolor='black', gridcolor=GRIDCOLOR, zeroline=False)
fig_2d.show()

fig_3d = px.scatter_3d(
    df_pca, x='PC1', y='PC2', z='PC3', color='species', color_discrete_map=SPECIES_COLORS,
    category_orders={'species': SPECIES_ORDER}, template=TEMPLATE
)
fig_3d.update_traces(marker=dict(size=5, opacity=0.8), marker_line=dict(width=0))
fig_3d.update_layout(title='Interactive 3D Dimensionality Scaling mapped across PC1, PC2, and PC3', height=500, margin=dict(t=50, b=0))
fig_3d.show()

# ── PCA Loadings View ──────────────────────────────────────
display(Markdown("### PCA Feature Loadings"))
loadings = pd.DataFrame(
    pca.components_[:2].T, columns=['PC1', 'PC2'],
    index=['Bill Length', 'Bill Depth', 'Flipper Length', 'Body Mass']
)
display(loadings.round(3).style.background_gradient(cmap='coolwarm', axis=None).set_table_styles([{'selector': 'th', 'props': [('background-color', '#15478A'), ('color', 'white')]}]))

insight_box("Adding a 3rd Principal Component validates depth, but the pure 2D split already successfully isolates the exact partitions we identified manually. "
            f"PC1 independently controls {var1:.1f}% of variance, heavily anchored by generalized Body Mass.")

### PCA Feature Loadings

,PC1,PC2
Bill Length,0.454000,0.600000
Bill Depth,-0.399000,0.796000
Flipper Length,0.577000,0.006000
Body Mass,0.550000,0.076000


In [19]:
# ── K-Means Interactive Projection ─────────────────────────
def plot_kmeans(k=3):
    clear_output(wait=True)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    df_cluster = df_pca.copy()
    df_cluster['Cluster'] = [f'Cluster {c}' for c in clusters]
    cluster_palette = px.colors.qualitative.Set2[:k]

    fig = make_subplots(rows=1, cols=2, subplot_titles=[f'K-MEANS (k={k})', 'ACTUAL GROUND TRUTH'])
    for i in range(k):
        mask = df_cluster['Cluster'] == f'Cluster {i}'
        fig.add_trace(go.Scatter(x=df_cluster.loc[mask, 'PC1'], y=df_cluster.loc[mask, 'PC2'], mode='markers', marker=dict(color=cluster_palette[i], size=8, opacity=0.7), name=f'Cluster {i}', showlegend=False), row=1, col=1)
    for sp in SPECIES_ORDER:
        mask = df_cluster['species'] == sp
        fig.add_trace(go.Scatter(x=df_cluster.loc[mask, 'PC1'], y=df_cluster.loc[mask, 'PC2'], mode='markers', marker=dict(color=SPECIES_COLORS[sp], size=8, opacity=0.7), name=sp, showlegend=False), row=1, col=2)

    ari = adjusted_rand_score(df['species'], clusters)
    
    fig.update_layout(
        **LAYOUT_DEFAULTS,
        title=dict(text=f'K-MEANS ALGORITHM ACHIEVES {ari:.2f} AGREEMENT WITH NATURE<br>'
                        '<sup style="color:#505050;">Adjusted Rand Index mapping correlation</sup>'),
        height=450, margin=dict(b=80), annotations=[source_annotation()]
    )
    for c in [1, 2]:
        fig.update_xaxes(title_text='PC1', showgrid=True, gridcolor=GRIDCOLOR, row=1, col=c)
        fig.update_yaxes(title_text='PC2', showgrid=True, gridcolor=GRIDCOLOR, row=1, col=c)
    fig.show()

display(Markdown("### Adjust the number of clusters to simulate the algorithm trying to find the tribes:"))
k_slider = widgets.IntSlider(value=3, min=2, max=8, step=1, description='Clusters (k):')
widgets.interact(plot_kmeans, k=k_slider)

### Adjust the number of clusters to simulate the algorithm trying to find the tribes:

interactive(children=(IntSlider(value=3, description='Clusters (k):', max=8, min=2), Output()), _dom_classes=(…

<function __main__.plot_kmeans(k=3)>

In [20]:
# ── Validation: Elbow Method & Silhouette Scores ───────────
inertias, sil_scores = [], []
K_range = list(range(2, 9))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    preds = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, preds))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score (Compactness)'], horizontal_spacing=0.08)

# Elbow Trace
fig.add_trace(go.Scatter(x=K_range, y=inertias, mode='lines+markers', marker=dict(size=8, color='#15478A'), line=dict(width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=[3], y=[inertias[1]], mode='markers+text', marker=dict(size=14, color='#3B5BBE', symbol='star'), text=['Optimal (k=3)'], textposition='top right', textfont=dict(size=12, color='#3B5BBE', family='Arial Bold'), showlegend=False), row=1, col=1)

# Silhouette Trace
fig.add_trace(go.Scatter(x=K_range, y=sil_scores, mode='lines+markers', marker=dict(size=8, color='#5BB5D5'), line=dict(width=2.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=[2], y=[sil_scores[0]], mode='markers+text', marker=dict(size=14, color='#876EC4', symbol='circle'), text=['Peak (k=2)'], textposition='top right', textfont=dict(size=12, color='#876EC4', family='Arial Bold'), showlegend=False), row=1, col=2)

fig.update_layout(
    **LAYOUT_DEFAULTS,
    title=dict(text='TRIANGULATING OPTIMAL BUNDLES THROUGH INERTIA AND SILHOUETTE<br>'
                    '<sup style="color:#505050;">Secondary scoring guarantees mathematical rigidity</sup>'),
    height=350, showlegend=False, margin=dict(b=80), annotations=[source_annotation()]
)
fig.update_xaxes(showgrid=True, gridcolor=GRIDCOLOR, dtick=1, title="Number of Clusters (k)")
fig.update_yaxes(showgrid=True, gridcolor=GRIDCOLOR)
fig.show()

insight_box("While the Elbow Method naturally confirms 3 clusters as the optimal grouping, the Silhouette Score peaks strongly at 2. "
            "This occurs because Chinstrap and Adelie bodily structures are highly homologous natively, tricking silhouette isolation algorithms. Nature's 3 wins out.")

---
# Epilogue: What the Data Tells Us

The story of the Palmer Archipelago penguins is ultimately one of **adaptation, specialization, and nature's elegant engineering:**

| Act | Discovery | Key Takeaway |
|:---|:---|:---|
| **1. The Territory** | Geographic isolation | Adelies are pioneers; Gentoos & Chinstraps are specialists |
| **2. The Paradox** | Simpson's Paradox | Aggregated data can be deeply misleading — always look within groups |
| **3. The Engine** | Flipper-Mass correlation | Form follows function — the strongest correlation in the dataset |
| **4. The Divide** | Sexual dimorphism | Males outsize females consistently, reflecting evolutionary pressures |
| **5. The Machine** | Unsupervised clustering | Algorithms rediscover nature's groupings from raw measurements alone |

> *"The data tells a story of three tribes, shaped by millions of years of evolution, each perfectly adapted to their niche in one of Earth's harshest environments."*

---
*Source: Palmer Penguins Dataset (Horst, Hill & Gorman, 2020) | Built with Python, Plotly, and Scikit-learn for COMP4010, VinUniversity (Spring 2026)*